# 03 Using Tools
In this workbook we'll discuss how to integrate tools with your models.

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
from langchain.tools import tool

# NOTE: This is a simple example of how to create a tool.
# In practice, you would likely want to create more complex
# tools that can perform a variety of tasks.
# The important thing is that:
#  - You annotate a tool with the @tool decorator
#  - You MUST define a doc comment for the function (Google style)


@tool
def square_root(num: float) -> float:
    """returns the square root of a number

    Args:
        num (float): number whose square root is desired
    Returns:
        (float): the square root of the number
    """
    print(f"------ Calling square_root({num}) tool ------")
    return num**0.5

In [3]:
# you can invoke a too directly - note how params are passed
response = square_root.invoke({"num": 4156})
print(response)

------ Calling square_root(4156.0) tool ------
64.4670458451448


In [4]:
# you can add additional info to the tool decorator


@tool("square_root", description="Calculate the square root of a number")
def tool1(num: float) -> float:
    """returns the square root of a number

    Args:
        num (float): number whose square root is desired
    Returns:
        (float): the square root of the number
    """
    return num**0.5

In [5]:
response = tool1.invoke({"num": 4156})
print(response)

64.4670458451448


### Adding tools to agent
Here is how you can add tools to an agent

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5-nano",
    system_prompt="""You are a math wizard that can preform calculations 
        using tools provided to you. You have the following tools:
        - square_root - to calculate square root of number.
        Always use tools first before relying on your own ability""",
    tools=[square_root],
)

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Calculate square root of 4156",
            }
        ]
    }
)
print(response["messages"][-1].content)

------ Calling square_root(4156.0) tool ------
The square root of 4156 is approximately 64.4670458451448.

If you want it rounded:
- to 4 decimals: 64.4670
- to 2 decimals: 64.47


Notice that the response is not structured. I would like to get just the funal result - 64.4670458451448, here is where I'll used structured output

In [7]:
from pydantic import BaseModel


class MathResult(BaseModel):
    result: float


agent2 = create_agent(
    model="openai:gpt-5-nano",
    system_prompt="""You are a math wizard that can preform calculations 
        using tools provided to you. You have the following tools:
        - square_root - to calculate square root of number.
        Always use tools first before relying on your own ability""",
    tools=[square_root],
    response_format=MathResult,
)

response = agent2.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Calculate square root of 4156",
            }
        ]
    }
)
print(f"Result -> {response["structured_response"].result:.3f}")

------ Calling square_root(4156.0) tool ------
Result -> 64.467


### Add a web search tool using Tavily Search
In this section, we'll see a specific case of using web-search with an agent. Specifically, we'll use Tavily search.

**NOTE:** To enable Tavily Search, do the following:
1. Install Tavily search modules: `uv add tavily` && `uv add langchain_tavily`  
2. Get Tavily Search API key [here](https://app.tavily.com/home) - look for API Keys section on web page.
3. Add TAVILY_API_KEY to your `.env` file.

But first, let's see how it works _without_ the web search tool.

In [8]:
from pydantic import BaseModel


class MathOrSearchResult(BaseModel):
    # structured response
    result: str


agent3 = create_agent(
    model="openai:gpt-5-nano",
    # using RCTF (Role, Context, Tools, Format) prompting framework
    system_prompt="""You are an assistant that can preform math calculations
        and respond to other queries. 
        You have access to the following tools:
        - square_root - to calculate square root of number.
        Depending on user's query, you may use the appropriate tool.
        Always use tools first before relying on your own ability. 
        Return a crisp response based on the tool you used. If you use web search, 
        return the most relevant result.
    """,
    # NOTE: not giving it web-search capability yet!
    tools=[square_root],
    response_format=MathOrSearchResult,
)

response = agent3.invoke(
    {
        "messages": [
            {
                "role": "user",
                # let's be very specfic in our query!
                "content": "Who is the chief minister of Maharashtra as of April 2026?",
            }
        ]
    }
)
print(f"Result -> {response["structured_response"].result}")

Result -> I don’t have real-time data. My knowledge only goes up to June 2024, and I can’t verify who was the Chief Minister of Maharashtra in April 2026. As of mid-2024, the Chief Minister was Eknath Shinde. Please check a current source (official Maharashtra government site or reputable news outlets) for the exact name in April 2026. If you’d like, I can guide you on how to quickly verify this or summarize the recent CM timeline.


Now let's add a `web_search` tool to add web search capabilities to our agent.

In [9]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()


@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information
    Args:
        query (str): the search query
    Returns:
        Dict[str, Any]: the search results
    """
    print(f"------ Calling web_search({query}) tool ------")
    return tavily_client.search(query)


# calling the tool directly
response = web_search.invoke("Who is the current mayor of San Francisco?")
response

------ Calling web_search(Who is the current mayor of San Francisco?) tool ------


{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Mayor_of_San_Francisco',
   'title': 'Mayor of San Francisco - Wikipedia',
   'content': 'The current mayor is Democrat Daniel Lurie.',
   'score': 0.95334905,
   'raw_content': None},
  {'url': 'https://ballotpedia.org/Daniel_Lurie',
   'title': 'Daniel Lurie - Ballotpedia',
   'content': 'Daniel Lurie is the Mayor of San Francisco in California. He assumed office on January 8, 2025. His current term ends on January 8, 2029.',
   'score': 0.8877607,
   'raw_content': None},
  {'url': 'https://apnews.com/article/san-francisco-new-mayor-liberal-city-81ea0a7b37af6cbb68aea7ef5cc6a4f0',
   'title': "San Francisco's new mayor is starting to unite the fractured city",
   'content': 'San Francisco Mayor Daniel Lurie, a political newcomer and Levi Strauss heir, has marked his first 100 days with a hands-on, business-friendly a

In [10]:
# now let's enable web search & math calculation in the same agent

agent4 = create_agent(
    model="openai:gpt-5-nano",
    # using RCTF (Role, Context, Tools, Format) prompting framework
    system_prompt="""You are an assistant that can preform math calculations
        and respond to other queries. 
        You have access to the following tools: 
        - square_root - to calculate square root of number.
        - web_search - to search the web for information.
        Depending on user's query, you may use the appropriate tool.
        Always use tools first before relying on your own ability.         
        Return a crisp response based on the tool you used. If you use web search, 
        return the most relevant result. For square root, return just the calculation result.
    """,
    tools=[square_root, web_search],
    response_format=MathOrSearchResult,
)

# math calculation
response = agent4.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Calculate square root of 4156",
            }
        ]
    }
)
print(f"Result -> {float(response['structured_response'].result):.3f}")

# web search
response = agent4.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Who is the chief minister of Maharashtra as of April 2026?",
            }
        ]
    }
)
print(f"Result -> {response['structured_response'].result}")

------ Calling square_root(4156.0) tool ------
Result -> 64.467
------ Calling web_search(Chief Minister of Maharashtra April 2026) tool ------
Result -> Devendra Fadnavis (Chief Minister of Maharashtra as of April 2026).


Perfect!

In this notebook you saw how to use tools to enhance the capability of a basic agent. Tools help agents do work that they are not intrinsically capable of doing - for example, getting the _latest_ information from the web.